# Grand Ethiopian Renaissance Dam — 4x5 Optimization 

This notebook is a worked example for a multi-objective optimization
for an artificial-reef design problem. It shows how to:

- Define design variables and bounds
- Specify multiple objective functions and preference mappings
- Build constraints
- Run a Genetic Algorithm (GA) using two (potentially three) aggregation paradigms
- Visualize preference functions and optimization results


## Problem

For many years, Ethiopia has faced a significant shortage of reliable electricity, limiting economic development and access to essential services, particularly in rural areas. At the same time, the country has substantial hydropower potential due to its extensive river systems.
To address this energy shortage, several hydropower development options were considered, including multiple smaller dams at different locations along the Nile and its tributaries. The Grand Ethiopian Renaissance Dam (GERD) was ultimately selected because its location on the Blue Nile provides favourable conditions for large-scale hydropower generation and water storage. Its large reservoir and high elevation difference allow it to generate a substantial amount of electricity from a single project.

However, the Nile is a transboundary river and is a crucial water source for downstream countries, particularly Sudan and Egypt. Changes in the timing and amount of water flowing downstream can therefore have significant implications for water supply, agriculture, and hydropower generation in these countries. This makes the GERD not only an Ethiopian energy project, but also an important regional water-management issue.

In addition to electricity generation, the GERD could provide potential benefits such as more regulated downstream flows, improved water management during dry periods, and opportunities for electricity exports. These additional functions make the GERD a potentially multipurpose infrastructure project, while also creating the need for cooperation between the countries sharing the Nile.



## Importing Required Packages

Below, the required packages are imported. To run this notebook successfully, your environment must include the required dependencies. The virtual environment provided in MUDE, called **`mude-base`**, is sufficient for this purpose.

In addition, several plotting settings are defined, and the local optimisation code is imported. It is important that the provided zip file is extracted in its entirety, as the structure of the directory is required for the imports to work correctly.

The local module `genetic_algorithm_pfm` is imported using a **relative import**, which depends on the directory structure of the project. When the notebook is executed within the provided directory structure, this will work as intended. However, running the notebook outside of this directory will result in the following error: `ModuleNotFoundError: No module named 'genetic_algorithm_pfm'`

In [2]:
# Import libraries
import matplotlib.pyplot as plt
import numpy as np
from scipy.interpolate import pchip_interpolate
from scipy.optimize import minimize 

# Define default plotting parameters
plt.rcParams['font.size'] = '10'  
plt.rcParams['savefig.dpi'] = 300  

# Import local module for genetic algorithm
from genetic_algorithm_pfm import GeneticAlgorithm

## Design Variables and Bounds

The artificial reef geometry is parameterized using four continuous design variables:

| Variable | Description | Unit | Type |
|----------|-------------|------|------|
| `x1` | Dam crest width | m | Continuous |
| `x2` | Dam height | m | Continuous |
| `x3` | Amount of turbines | - | Continuous |


**Note:** In the code, design variables are defined only by their type (discrete or continuous) and their bounds — *not* by their physical meaning. Before implementing, it is good practice to write down your choice of design variables on paper, including their units and expected ranges. This helps avoid errors and builds physical intuition for the problem.



Each variable is assigned bounds that reflect physically meaningful and feasible values for a real-world gravity dam design. These are defined below.

<img src="new_picture_design_GERD_variables.PNG" width="500">

`X_irl` contains a reference design based on a real-world installation, which will later be used for comparison against the optimised solutions. Its design variable values are:

In [3]:
# Define the names of variables for later use in plotting and analysis
design_variables = (
    ('x1', 'Dam crest width', 'm'),
    ('x2', 'Dam height', 'm'),
    ('x3', 'Turbine flow capacity', 'm³ /s'),
)

# set bounds for all variables
b1 = [20,200]       # x1 crest at the bottom
b2 = [50, 250]      # x2 heigth - starting at a profitable height for energy generation
b3 = [1, 25]        # amount of turbines
#b3 = [270, 6000]        # x3 turbine flow capacity from 1 turbine 
bounds = [b1, b2, b3]

X_irl = [10, 1455, 1000]
plot_irl = True  # Can be true or False, depending on whether you want to plot the IRL point or not

## Constraints

Constraints ensure that all solutions remain physically feasible. For this artificial reef model, one physical constraint is considered: **the reef has a minimum distance from the beach**. This constraint, `constraint_1`, is a simple geometric relation corresponding to the distance `x5` shown in the figure above. A constraint function may use as many or as few design variables as necessary.

Each *constraint function* must be expressed as an algebraic expression of the form:

$$g(\mathbf{x}) \leq 0 \quad \text{(feasible)}$$

> **Important:** The constraint function should return only the value of $g(\mathbf{x})$ — do *not* encode the inequality inside the function. The inequality type (`'ineq'`) is specified separately when registering the constraint in `cons`, and the optimiser handles the comparison.

The constraints are registered as follows:

```python
cons = [['ineq', constraint_1], ['ineq', constraint_2]]
```

In [ ]:
def constraint_1(variables):
    """Moment stability:

    :return: 1-D array (length n) with constraint values; arr>0 means violation for 'ineq' constraints.
    """
    x1 = variables[:, 0]
    x2 = variables[:, 1]
    wb = 100
    m_water = (1/6) * x2**3 * 10       # maximum moment of water on dam, in kN per meter dam (m^3 * kN/m^3)
    m_own_dam = 1/6 * wb**2 * x2 * 24   # own moment material dam in kN per meter dam (m^2 * m * kN/m^3)


    return -1 * (m_own_dam - (m_water * 1.5)) # < 0

con = [['ineq', constraint_1]]

## Objective Functions

This model optimises five objectives, each representing a different stakeholder identified during the preceding stakeholder analysis. The table below summarises which objective was chosen to represent each stakeholder.

| Stakeholder | Objective | Unit |
|-------------|-----------|------|
| Ethiopian government | Energy generation | kWh |
| Downstream governments & local residents | Watermanagement | m³ in water reservoir |
| Environmentalists | Disruption of ecosystems | CO2-emissions |


Each objective is then expressed as a function of the design variables. Where needed, physical constants are introduced to make the units consistent — reasonable estimates are sufficient.

In [ ]:
def objective_function_1(x1, x2, x3):
    """
    Energy generation function.

    :return: estimated energy generated in kWh.
    """
    # Physical constants
    eta = 0.85          # turbine efficiency [-]
    rho = 1000          # water density [kg/m3]
    g = 9.81            # gravitational acceleration [m/s2]
    Q = x3 * 270        # flow discharge 1 turbine = 270 m3/s
    t = 365 * 24 * 3600 # one year [s]
    # Head depends on design depth
    H = x2
    # Power generated [W]
    P = eta * rho * g * Q * H
    # Energy generated [kWh]
    E = P * t / 1000

    return E 
    
    

def objective_function_2(x1, x2, x3):
    """
    Volume water reservoir function.
    
    :return: float of volume of water stored in cubic meters.
    """
    cons9 = 1  # Area of water reservoir in m^2 (L * b)
    cons10 = x2      # depth of water reservoir in m
    cons11 = x1

    return (cons9 * cons10) + cons11


def objective_function_3(x1, x2, x3):
    """
    CO2-function
    
    :return: float of CO2-emission per volume material.
    """
    L = 1780                            # m
    wb = 100
    volume = x2 * ((wb * x1) / 2) * L    # height, width of base and crest, length in m   
    cons18 = volume                     # material used (volume) in m3
    cons19 =  200                       # Emission factor of a kg RCC (concrete) in m3

    return cons18 * cons19


# Define the list of objectives with their corresponding names and units and stakeholders for later use in plotting and analysis
objectives = [
    (objective_function_1, "Energy generation",      "kWh",              "Ethiopian government"),
    (objective_function_2, "Water management",       "m^3",               "Downstream governments & local residents"),
    (objective_function_3, "CO2-emmission",          "m^3",              "Environmentalists")]


In [ ]:
def Energy(H):          
        # Physical constants
        eta = 0.85          # turbine efficiency [-]
        rho = 1000          # water density [kg/m3]
        g = 9.81            # gravitational acceleration [m/s2]
        Q = 100             # discharge [m3/s]
        t = 365 * 24 * 3600 # one year [s]
        # Power generated [W]
        P = eta * rho * g * Q * H
        # Energy generated [kWh]
        E = P * t / 1000

        return E

print(Energy(10), Energy(100), Energy(250))
E1 = Energy(10)
E2 = Energy(100)
E3 = (Energy(250))
h = [10, 100, 250]
E = [E1, E2, E3]
# plt.plot(h, E)
# plt.title('Generated power as function of height')
# plt.xlabel('Height [m]')
# plt.ylabel('Generated Energy [kWh]')
# plt.grid()

While developing the objectuve functions, it can be helpful to investigate their behavior. Here the attainable minimum and maximum of each objective is computed. These ranges serve as a sanity check — if the values are physically unreasonable, revisit the formulation and its constants. The ranges also define the interval over which each objective is rated in the preference functions.

Optionally, the objective functions can be visualised using `matplotlib` or [Desmos](https://www.desmos.com/calculator) to further inspect their behaviour.

To compute the minimum and maximum values we can use the `minimize` command, which is part of the SciPy package. For more infomration see: https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.minimize.html.

In [ ]:
# Finding min and max for each objective using scipy's minimize function, starting from the midpoint of the bounds
objective_minmax = {} # Dictionary to store the min and max values for each objective       
midpoints = [np.mean(b) for b in bounds]

for idx, (obj_func, name, unit, stakeholder) in enumerate(objectives):
    wrapped = lambda x, sign=1: sign * obj_func(*x)  # obj_func accepts a single array-like X
    
    min_val =  minimize(wrapped, x0=midpoints, bounds=bounds, method='L-BFGS-B').fun
    max_val = -minimize(lambda x: wrapped(x, sign=-1), x0=midpoints, bounds=bounds, method='L-BFGS-B').fun

    objective_minmax[name] = min_val, max_val
    print(f"  Objective {idx+1}  :    min = {min_val:>15,.1f}  {unit:<12}    max = {max_val:>15,.1f}  {unit}")


## Weights

When combining multiple objectives into a single score, each objective is assigned a weight **$w_i$** reflecting its relative importance. All weights must sum to one:

$$\sum_{i=1}^{5} w_i = 1$$

| Weight | Stakeholder | Objective |
|--------|-------------|-----------|
| $w_1$ | Ethiopian government | Energy generation |
| $w_2$ | Downstream governments & local residents | measurements water reservoir |
| $w_3$ | Environmentalists | CO2-emission |
<!-- | $w_4$ | Local community | Safety |
| $w_5$ | Tourism | Economic growth | -->

In this model, all weights are initialised equally ($w_i = \frac{1}{5} = 0.2$) for simplicity.

In [ ]:
# Weights per stakeholder — must sum to 1. Before game all weights were set equal.
#                   kWh   water  CO2   
weights_before =    [0.33333333, 0.3333333, 0.333333333]
weights_after =     [0.75, 0.15, 0.10]

weights = weights_before  # set to which weight to use

# Verify the weights sum to 1 (within numerical tolerance)
assert np.isclose(sum(weights), 1.0), f"Weights must sum to 1, got {sum(weights)}"

## Preference Curves and Preference Functions

Each objective value is mapped to a preference score between 0 and 100 using a *preference curve*, which represents a stakeholder's preference towards their objective. This allows objectives with different units and scales to be compared and combined, where 0 indicates the least desirable outcome and 100 the most desirable.

The curves are defined by a set of control points interpolated using `pchip_interpolate` from the SciPy package, which produces smooth curves that pass exactly through each point. Only the shape of the matters — so no analytical function is needed. Creating preference curves using interpolation is a convenient and flexible approach. Start simple and increase complexity only if necessary; aim to capture the essential behaviour with as few points as possible. The prefence curves are then plotted.

For this reef model, four objectives are rated linearly in the technical cycle. The exception is sediment trapping (objective 3), which follows a non-linear preference: too little trapping fails to widen the beach, but too much blocks sediment from reaching downstream beaches — so an intermediate value is preferred.

Each preference curve is then combined with its corresponding objective function to form a *preference function*, which maps design variables directly to a preference score.

In [ ]:
# Define preferences    

prefs_before = [
    [np.array([3549999636000, 443749954500000]), np.array([0, 100])],         # preference energy generation for ethiopian govenment
    [np.array([70, 450]),                       np.array([0, 100])],          # preference water management for local resisdents and downstream governments
    [np.array([17800000000, 890000000000]),      np.array([100, 0])]          # preference co2-emission for environmentalists
]

# pref_after = [[[150000, 2000000, 4250000],      [100, 80, 0]],
#                 [[8, 120, 350],                 [0, 60, 100]],
#                 [[0, 80000, 150000, 375000],    [0, 100, 50, 0]]],


prefs = prefs_before  # set to which preferences to use


# Preference curve values for plotting
# Generate objective value ranges 
obj_vals_list = []
for _, name, _, _ in objectives:
    obj_vals = np.linspace(*objective_minmax[name])
    obj_vals_list.append(obj_vals)

# Generate preference values using interpolation
pref_vals_list = []
for pref, obj_vals in zip(prefs, obj_vals_list):
    pref_vals = pchip_interpolate(pref[0], pref[1], obj_vals)
    pref_vals_list.append(pref_vals)

# Plotting the preference curves 
fig, axes = plt.subplots(2, 3, figsize=(12, 8))

for ax, (obj_vals, p_vals), (_, name, unit, stakeholder) in zip(axes.flat, zip(obj_vals_list, pref_vals_list), objectives): # Loop to plot each preference curve in its own subplot
    ax.plot(obj_vals, p_vals, color='black')
    ax.set_xlim(min(obj_vals), max(obj_vals))
    ax.set_ylim(0, 100)
    ax.set_title(stakeholder)
    ax.set_xlabel(f'{name} [{unit}]')
    ax.set_ylabel('Preference score')
    ax.grid(linestyle='--')

for ax in axes.flat[len(objectives):]:
    ax.set_visible(False)  # hide empty subplot
fig.tight_layout()
# plt.savefig('preference_curves.png')  # Uncomment this line to save the figure as a PNG file
plt.show()


# Defines preference functions that convert raw objective values to 0-100 preference scores
def pref_func_1(x1, x2, x3):
    energy = objective_function_1(x1, x2, x3)
    return pchip_interpolate(prefs[0][0], prefs[0][1], energy)

def pref_func_2(x1, x2, x3):
    waterreservoir = objective_function_2(x1, x2, x3)
    return pchip_interpolate(prefs[1][0], prefs[1][1], waterreservoir)

def pref_func_3(x1, x2, x3):
    co2 = objective_function_3(x1, x2, x3)
    return pchip_interpolate(prefs[2][0], prefs[2][1], co2)

pref_funcs = [pref_func_1, pref_func_2, pref_func_3]  # list of preference functions for easier handling


def objective(variables):
    """
    Objective function that is fed to the GA. Calles the separate preference functions that are declared above.

    :param variables: array with design variable values per member of the population. Can be split by using array
    slicing
    :return: 1D-array with aggregated preference scores for the members of the population.
    """
    x1 = variables[:, 0]
    x2 = variables[:, 1]
    x3 = variables[:, 2]

    # calculate the preference scores
    p_1 = pref_func_1(x1, x2, x3)
    p_2 = pref_func_2(x1, x2, x3)
    p_3 = pref_func_3(x1, x2, x3)

    
    return weights, [p_1, p_2, p_3]

## Optimisation

Now that everything is in place, we can run the optimisation. For more details on the available configuration options, see the docstring of `GeneticAlgorithm` (accessible via `help()`). Note that a genetic algorithm relies on randomness, so it is expected that results will differ slightly between separate executions.

The optimisation can be run using three aggregation methods: `'minmax'`, `'a-fine'`, and optionally `'tetra'`. These methods differ in how the individual preference scores are combined into a group preference during the optimisation process; see `aggregations.md` for more information on how each method works and when to use it. These methods determine, mathematically, what is considered the "best" group preference. However, what is considered "best" is not purely a mathematical question, it also involves a qualitative judgement about which outcome is most appropriate for the problem at hand.

After computing the optimal results, the solutions are plotted on top of their corresponding preference curves.

In [ ]:
# We run the optimization with two paradigms
paradigm = ["minmax", "a-fine", "tetra"]
markers = ["o", "*", "^"]  # Added '^' for tetra
colours = ["orange", "green", "purple"]  # Added 'purple' for tetra

# Define the figure and axes before the loop
fig, axes = plt.subplots(2, 3, figsize=(12, 8))

# Plot underlying preference curves using a loop
for ax, o_vals, p_vals, (_, name, unit, stakeholder) in zip(axes.flat, obj_vals_list, pref_vals_list, objectives): 
    ax.plot(o_vals, p_vals, color='black', label='Preference curve')
    ax.set_xlim(min(o_vals), max(o_vals))
    ax.set_ylim(0, 100)
    ax.set_title(stakeholder)
    ax.set_xlabel(f'{name} [{unit}]')
    ax.set_ylabel('Preference score')
    ax.grid(linestyle='--')

cons = []

for i in range(len(paradigm)):
    # Dictionary with parameter settings for the GA run with the IMAP solver
    options = {
        'n_bits': 8,
        'n_iter': 400,
        'n_pop': 500,
        'r_cross': 0.8,
        'max_stall': 16,
        'aggregation': paradigm[i],  
        'var_type': 'real'
    }

    # Run the GA and print its result
    print(f'\nRun GA with {paradigm[i]}')
    ga = GeneticAlgorithm(objective=objective, constraints=cons, bounds=bounds, options=options)
    _, optimal_design_var, _ = ga.run()

    # print the optimal design variable values
    print(f'Optimal design variable values using {paradigm[i]}:') 
    for opt_var, (xi, name, unit) in zip(optimal_design_var, design_variables):
        print(f'  {xi} : {name:<25} = {opt_var:>10.2f} {unit}') 

    # Print and plot the optimal objective results and stakeholder preferences
    print(f'Optimal objective results and stakeholder preference using {paradigm[i]}:')
    for j, (ax, (obj_func, name, unit, stakeholder)) in enumerate(zip(axes.flat, objectives)):
        obj_value = obj_func(*optimal_design_var) # Calculate resulting objective value
        pref_score = pref_funcs[j](*optimal_design_var) # Calculate resulting preference score
    
        # Print and plot the optimal point on the preference curve
        print(f'  {name:<20} = {obj_value:>15,.2f} {unit:<10} -> {stakeholder:<20} pref: {pref_score:>6.2f}')
        ax.scatter(obj_value, pref_score, color=colours[i], marker=markers[i], s=50,
                   label=f'Optimal ({paradigm[i]})')
    
# plot the IRL design point if plot_irl is True
if plot_irl:   
    irl_obj_values = [obj_func(*X_irl) for obj_func, _, _, _ in objectives]
    irl_pref_scores = [pref_funcs[j](*X_irl) for j in range(len(objectives))]
    for j, ax in enumerate(axes.flat[:len(objectives)]):
            ax.scatter(irl_obj_values[j], irl_pref_scores[j], color='red', marker='s', s=50, label='IRL design')

for ax in axes.flat[:len(objectives)]:
    ax.legend(fontsize=10)

for ax in axes.flat[len(objectives):]:
    ax.set_visible(False)  # hide the empty subplot


# Adjust the layout 
fig.tight_layout()

# Save figure
# plt.savefig('optimal_results.png')  # Uncomment this line to save the figure as a PNG file

# Display the plot
plt.show()